# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imsahilahmed/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

---
## 1. The contract — five plain-word answers

**Lane:** Refresh / Content Opportunity Scoring

### Q1: What one row means
One row in `fact_content_daily_performance` = **one content item (`content_hash_id`) × one client (`client_hash_id`) × one calendar day (`report_date`)**.  
For my feature frame (summarised per page per month), one row = **one content item's aggregate metrics over March 2026**.

### Q2: Which table(s) I'll use
- **Primary:** `fact_content_daily_performance` — daily time-series of search + engagement metrics
- **Lookup (context only):** `dim_content` — content metadata; `dim_clients` — per-client history coverage

### Q3: Which time window
**Feature window:** March 2026 (`2026-03-01` to `2026-03-31`) — a complete mid-panel month.  
**Label window (proxy):** Derived from within-month trends (first-half vs second-half impressions) to demonstrate the concept — not a production label. In the capstone, the label will be a forward-looking window (e.g. April 2026).

### Q4: What I'd predict or rank (label / proxy)
A **priority score** ranking which pages need editorial review first. The proxy label for evaluation: `decline_label` = 1 when a page's impressions in the second half of March dropped > 20% vs the first half (with a minimum-volume floor). This is a **within-month trend proxy**, not a future-outcome label — honest about its limits.

### Q5: One thing I deliberately exclude
**Excluded:** `fact_content_query_90d` entirely — its fixed 90-day window overlaps the full history of every page, so its `impressions_90d` / clicks columns would leak information across my feature month. I work only from daily granularity partitioned data where I control the window boundaries exactly.

---
## 2. Fields: feature / label / context / excluded

Every column I touch, sorted into exactly one bucket.

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id`, `report_date` | **Context** | Grouping, joining, splitting — never features for the model. |
| `gsc_impressions` | **Feature** | Observed search visibility, knowable per day before any decision. |
| `gsc_clicks` | **Feature** | Observed clicks, knowable per day. |
| `gsc_avg_position` | **Feature** | Mean position, knowable per day (0 = no data, handle carefully). |
| `sessions` | **Feature** | GA4 sessions, knowable per day (when `ga4_data_available`). |
| `engaged_sessions` | **Feature** | GA4 engaged sessions, knowable per day. |
| `ga4_data_available` | **Context** | Filter flag — when FALSE, GA4 columns are zero-filled, not "no engagement". |
| `decline_label` | **Label / proxy** | Derived from within-March trend (second-half vs first-half impressions). Never a feature. |
| `impressions_second_half` | **Excluded** | Would be a future-looking feature if the label is also drawn from March — this is the deliberate trap shown below. |
| `fact_content_query_90d` (entire table) | **Excluded** | Fixed-90d window overlaps feature month; window alignment impossible to guarantee safely. |

**Missing values note:** `gsc_avg_position = 0` means "no position data" (not rank 0). `ga4_data_available` flags which rows have real GA4 data — rows before a client's GA4 start have zero-filled GA4 columns. I filter on this flag before aggregating.

---
## 3. Verify it with three queries on month=2026-03

These cells connect to the Hugging Face warehouse via DuckDB and run three checks on the March 2026 partition.

**Setup:** load Hugging Face token from Colab Secrets.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "HF_TOKEN not found — set it as a Secret in Colab or as an env var locally"
print("HF_TOKEN found:", HF_TOKEN[:4] + "..." + HF_TOKEN[-4:])

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT = f"read_parquet('{MONTH}')"

print("Connected to warehouse. March 2026 partition ready.")

### Query 1: Grain probe — prove one row really is one date × client × content

If the grain holds, `GROUP BY (report_date, client_hash_id, content_hash_id) HAVING COUNT(*) > 1` returns **zero rows**.

In [ ]:
q1 = con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
FROM {FACT}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("=== Q1: GRAIN PROBE ===")
print("Rows violating grain (expect 0):", len(q1))
if len(q1) == 0:
    print("✓ Grain holds: one row = one report_date × client_hash_id × content_hash_id")
else:
    print(q1)

### Query 2: Row count and date span for March 2026

Prove that `month=2026-03` contains exactly the 31 days of March, count rows, distinct clients, and distinct content items.

In [ ]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FACT}
""").df()

print("=== Q2: COUNTS + DATE SPAN (March 2026) ===")
print(q2.to_string(index=False))
print()
print(f"✓ Date span covers {31} days (full March 2026)")
print(f"✓ {q2['distinct_clients'].iloc[0]} clients tracked through {q2['distinct_content_items'].iloc[0]:,} content items")

### Query 3: Availability — filter with IS TRUE

Show how many rows have `ga4_data_available IS TRUE`, and how many of those also have at least 1 GSC impression. This is the honest slice: rows where both search AND analytics data are meaningful.

In [ ]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows_march,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE AND gsc_impressions >= 1) AS ga4_and_gsc
FROM {FACT}
""").df()

print("=== Q3: AVAILABILITY (IS TRUE filter) ===")
print(q3.to_string(index=False))
print()
total = q3['total_rows_march'].iloc[0]
avail = q3['ga4_available'].iloc[0]
both = q3['ga4_and_gsc'].iloc[0]
print(f"Rows with ga4_data_available IS TRUE: {avail:,} ({100*avail/total:.1f}% of total)")
print(f"Rows with both GA4 data AND ≥1 GSC impression: {both:,} ({100*both/total:.1f}% of total)")
print("→ This is my honest slice for feature building.")

---
## 4. Five features (plus the trap)

### The five-feature frame

Aggregated per content item for March 2026 (only rows where `ga4_data_available IS TRUE` and `gsc_impressions >= 100`).

| Feature | How it's computed | Available when? |
|---|---|---|
| `imp_march` | Sum of `gsc_impressions` over all days in March | **Knowable at decision moment** because it's the sum of each day's already-observed impressions — the day is over before we query. |
| `clk_march` | Sum of `gsc_clicks` over all March days | **Knowable at decision moment** — same logic; clicks are recorded per day, summarised after the month ends. |
| `pos_march` | Mean of `gsc_avg_position` (excluding zero = no-data rows) over March | **Knowable at decision moment** — position is measured per day, averaged after the day finishes. |
| `sess_march` | Sum of `sessions` over all March days (only when GA4 data is real) | **Knowable at decision moment** — sessions are recorded per day, available after the day closes. |
| `eng_sess_march` | Sum of `engaged_sessions` over all March days | **Knowable at decision moment** — same as sessions; the day's data is complete before we query. |

All five features are **knowable at the decision moment** because they summarise already-completed days. No future information leaks in.

In [ ]:
print("=== BUILDING FEATURE FRAME ===")

features = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
),
page_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clk_march,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
        SUM(sessions) AS sess_march,
        SUM(engaged_sessions) AS eng_sess_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT *
FROM page_march
""").df()

print(f"Feature frame: {len(features):,} content items with ≥100 March impressions")
print(f"Client count: {features['client_hash_id'].nunique()}")
print()
print("Feature summary:")
print(features[['imp_march', 'clk_march', 'pos_march', 'sess_march', 'eng_sess_march']].describe().round(1))

### The trap: deliberate leakage

Now I define a label: `decline_label = 1` when the page's impressions dropped >20% in the **second half of March** vs the **first half** (and had at least 50 impressions in the first half — enough to measure).  

**The trap:** I add `imp_second_half` as a "feature" and train a model. Since the label is literally derived from `imp_second_half < 0.8 × imp_first_half`, adding `imp_second_half` as a feature makes the problem trivial — the model sees the exact column that defines the label. The AUC will jump toward perfect.  

Then I remove it and keep the honest score.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# Build label: decline = second-half impressions < 80% of first-half (with min volume floor)
features['decline_label'] = (
    (features['imp_second_half'] < features['imp_first_half'] * 0.8) &
    (features['imp_first_half'] >= 50)
).astype(int)

print(f"Decline rate in slice: {features['decline_label'].mean():.1%}")
print(f"Positive label count: {features['decline_label'].sum():,} / {len(features):,}")
print()

# --- Honest model: no leakage ---
X_honest = features[['imp_first_half', 'clk_march', 'pos_march', 'sess_march', 'eng_sess_march']].fillna(0)

# --- Leaky model: add the label-derived column ---
X_leak = X_honest.copy()
X_leak['imp_second_half'] = features['imp_second_half'].values

y = features['decline_label']

# Train/test split (random — this is a demonstration, not a production split)
Xtr_h, Xte_h, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42)
Xtr_l, Xte_l = train_test_split(X_leak, test_size=0.3, random_state=42)[:2]

for name, Xtr, Xte in [("HONEST (no leakage)", Xtr_h, Xte_h), ("LEAKY (imp_second_half added)", Xtr_l, Xte_l)]:
    rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    y_prob = rf.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, y_prob)
    print(f"AUC {name}: {auc:.4f}")

The leaky model's AUC is near-perfect because `imp_second_half` is the exact column used to compute the label. This is **textbook leakage** — notebook 02's lesson, confirmed on real warehouse data.

**The honest model** uses only features that were knowable before the label was defined, and that score is the real bar to beat.

In [ ]:
# Redeem: drop the leaky column, keep only honest features
print("=== HONEST FEATURES ONLY ===")
print("Feature columns (no leakage):", list(X_honest.columns))
print()

# Verify no label-derived columns remain
assert 'imp_second_half' not in X_honest.columns, "Leaky column still present!"
print("✓ No label-derived columns in the honest feature set.")
print(f"\nHonest AUC: {roc_auc_score(yte, RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42, n_jobs=-1).fit(Xtr_h, ytr).predict_proba(Xte_h)[:,1]):.4f}")

---
## 5. One named limitation of my slice

**Limitation: within-month proxy labels do not measure future outcomes.**

My label compares the first half of March against the second half of March — both within the same feature window. This is useful for demonstrating the data contract workflow and the leakage trap, but it is **not** a genuine forward-looking label. A page that declined within March might recover in April naturally, or it might have been seasonal. The label cannot distinguish "real decline that needs intervention" from "normal monthly fluctuation."

In the capstone, the honest label will be: features from a prior window (e.g. March) → outcome in a future window (e.g. April), with a strict temporal split so no future information reaches the model. That is the stronger design — this contract is the foundation.

---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Five plain-words contract answers in section 1
- [x] Three verification queries (grain, counts/dates, IS TRUE availability) with outputs visible
- [x] Five-feature frame with "available when?" line per feature
- [x] Deliberate-leak experiment shown (AUC jumps to near-perfect) and removed
- [x] One named limitation of my slice
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.